In [53]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, TensorDataset 

from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

import scipy
import warnings

import xgboost


In [54]:
device = torch.device('cuda') if torch.cuda.is_available else torch.device('cpu')
np.random.seed(42)
warnings.simplefilter('ignore')

In [55]:
df_train_first = pd.read_csv('./data/train.csv')
df_val = pd.read_csv('./data/test.csv')

In [56]:
df_train, df_test = train_test_split(df_train_first, test_size=0.2, random_state=42)

## Data shape Before Feature Engineering

In [57]:
print('-- Training data shape before Feature Engineering', df_train.shape)
print('-- Testing data shape before Feature Engineering', df_test.shape)
print('-- Validation data shape before Feature Engineering', df_val.shape)

-- Training data shape before Feature Engineering (351312, 16)
-- Testing data shape before Feature Engineering (87828, 16)
-- Validation data shape before Feature Engineering (188165, 15)


In [ ]:
def feature_engineering(df, is_train=True, stats=None):
    df = df.copy()

    # ── 1. Compound encoding ──────────────────────────────────────────────────
    compound_life = {'SOFT': 20, 'MEDIUM': 30, 'HARD': 40, 'INTER': 25, 'WET': 15}
    df['expected_tyre_life'] = df['Compound'].map(compound_life).fillna(25)
    df['tyre_life_pct']      = df['TyreLife'] / df['expected_tyre_life']
    df['tyre_overdue']       = (df['TyreLife'] > df['expected_tyre_life']).astype(int)
    df['tyre_life_sq']       = df['TyreLife'] ** 2
    df['tyre_life_cubed']    = df['TyreLife'] ** 3

    # compound ordinal (grip level)
    compound_order = {'SOFT': 4, 'MEDIUM': 3, 'HARD': 2, 'INTER': 1, 'WET': 0}
    df['compound_ord'] = df['Compound'].map(compound_order).fillna(2)

    # ── 2. Stint features ─────────────────────────────────────────────────────
    df['stint_lap_sq']     = (df['TyreLife'] * df['Stint']) ** 2
    df['high_stint']       = (df['Stint'] >= 3).astype(int)  # 3rd stint = likely last

    # ── 3. Race progress features ─────────────────────────────────────────────
    # RaceProgress already exists — build on it
    df['in_early_race']    = (df['RaceProgress'] < 0.2).astype(int)
    df['in_pit_window']    = df['RaceProgress'].between(0.3, 0.7).astype(int)
    df['in_late_race']     = (df['RaceProgress'] > 0.8).astype(int)
    df['progress_x_tyre']  = df['RaceProgress'] * df['TyreLife']  # interaction

    # ── 4. Lap time features ──────────────────────────────────────────────────
    # LapTime_Delta already exists — build on it
    df['is_slowing_down']  = (df['LapTime_Delta'] > 0).astype(int)
    df['delta_sq']         = df['LapTime_Delta'] ** 2
    df['big_slowdown']     = (df['LapTime_Delta'] > df['LapTime_Delta'].quantile(0.75)).astype(int)

    # rolling stats per driver per race
    df = df.sort_values(['Race', 'Driver', 'LapNumber'])

    grp = df.groupby(['Race', 'Driver'])
    df['rolling_lap_3']    = grp['LapTime (s)'].transform(lambda x: x.rolling(3, min_periods=1).mean())
    df['rolling_lap_5']    = grp['LapTime (s)'].transform(lambda x: x.rolling(5, min_periods=1).mean())
    df['rolling_delta_3']  = grp['LapTime_Delta'].transform(lambda x: x.rolling(3, min_periods=1).mean())
    df['lap_vs_rolling']   = df['LapTime (s)'] - df['rolling_lap_3']
    df['lap_vs_rolling5']  = df['LapTime (s)'] - df['rolling_lap_5']

    # lag features
    df['prev_lap_time']    = grp['LapTime (s)'].shift(1)
    df['prev_delta']       = grp['LapTime_Delta'].shift(1)
    df['prev_position']    = grp['Position'].shift(1)
    df['prev_degradation'] = grp['Cumulative_Degradation'].shift(1)

    # ── 5. Position features ──────────────────────────────────────────────────
    df['Position_Change_abs'] = df['Position_Change'].abs()
    df['is_gaining']          = (df['Position_Change'] < 0).astype(int)  # moving up
    df['is_losing']           = (df['Position_Change'] > 0).astype(int)  # dropping back
    df['in_points']           = (df['Position'] <= 10).astype(int)
    df['is_leader']           = (df['Position'] == 1).astype(int)
    df['position_sq']         = df['Position'] ** 2  # nonlinear position effect

    # ── 6. Degradation features ───────────────────────────────────────────────
    df['deg_per_lap']         = df['Cumulative_Degradation'] / (df['TyreLife'] + 1)
    df['deg_x_tyre_life']     = df['Cumulative_Degradation'] * df['TyreLife']
    df['deg_acceleration']    = grp['Cumulative_Degradation'].transform(
                                    lambda x: x.diff().diff())  # rate of rate of degradation

    # ── 7. Stint x Compound interactions ─────────────────────────────────────
    df['stint_x_compound']    = df['Stint'] * df['compound_ord']
    df['tyrelife_x_compound'] = df['TyreLife'] * df['compound_ord']
    df['deg_x_compound']      = df['Cumulative_Degradation'] * df['compound_ord']

    # ── 8. Driver stats (fit on train, apply to test) ─────────────────────────
    if is_train:
        stats = {}

        stats['driver_pit_rate'] = df.groupby('Driver')['PitStop'].mean()

        stats['driver_compound_rate'] = df.groupby(
            ['Driver', 'Compound'])['PitStop'].mean().unstack(fill_value=0)
        stats['driver_compound_rate'].columns = [
            f'driver_pit_{c.lower()}' for c in stats['driver_compound_rate'].columns]

        stats['driver_avg_tyre_life'] = df.groupby('Driver')['TyreLife'].mean()
        stats['driver_avg_deg']       = df.groupby('Driver')['Cumulative_Degradation'].mean()

        stats['race_pit_rate']        = df.groupby('Race')['PitStop'].mean()
        stats['year_pit_rate']        = df.groupby('Year')['PitStop'].mean()

    global_mean = df['PitStop'].mean() if is_train else 0.05

    df['driver_pit_rate']     = df['Driver'].map(stats['driver_pit_rate']).fillna(global_mean)
    # df['driver_avg_tyre']     = df['Driver'].map(stats['driver_avg_tyre_life']).fillna(df['TyreLife'].mean())
    df['driver_avg_deg']      = df['Driver'].map(stats['driver_avg_deg']).fillna(df['Cumulative_Degradation'].mean())
    df['race_pit_rate']       = df['Race'].map(stats['race_pit_rate']).fillna(global_mean)
    df['year_pit_rate']       = df['Year'].map(stats['year_pit_rate']).fillna(global_mean)

    df = df.join(
        stats['driver_compound_rate'],
        on='Driver'
    ).fillna(global_mean)

    # ── 9. Drop raw categoricals ──────────────────────────────────────────────
    df.drop(columns=['Driver', 'Compound', 'Race'], inplace=True)

    return (df, stats) if is_train else df


# ── Apply ─────────────────────────────────────────────────────────────────────
df_train_fe, stats = feature_engineering(df_train, is_train=True)
df_test_fe          = feature_engineering(df_test,  is_train=False, stats=stats)
df_val_fe           = feature_engineering(df_val,   is_train=False, stats=stats)

# ── Split features / target ──────────────────────────────────────────────────
drop_cols = ['id', 'PitStop']
X_train   = df_train_fe.drop(columns=drop_cols).values
y_train   = df_train_fe['PitStop'].values

X_test    = df_test_fe.drop(columns=drop_cols).values
y_test    = df_test_fe['PitStop'].values

df_val_processed = df_val_fe.drop(columns=['id'], errors='ignore').values

In [59]:
# Unscaled → for LGB, XGB, CatBoost, RF, ExtraTrees, GBM

X_train_tree = X_train
X_test_tree = X_test
X_val_tree = df_val_processed


# Scaled → for ANN, SVM, KNN, LR, MLP
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_teat_scaled = scaler.fit_transform(X_test)
X_val_scaled = scaler.fit_transform(df_val_processed)

## Data shape After Feature Engineering

In [60]:
print('-- Training data shape After Feature Engineering', X_train_tree.shape)
print('-- Testing data shape After Feature Engineering', X_test_tree.shape)
print('-- Validation data shape After Feature Engineering', X_val_tree.shape)

-- Training data shape After Feature Engineering (351312, 57)
-- Testing data shape After Feature Engineering (87828, 57)
-- Validation data shape After Feature Engineering (188165, 57)


## Building ANN

In [63]:
train_set = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
test_set  = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long))
val_set   = TensorDataset(torch.tensor(df_val_processed, dtype=torch.float32))

In [64]:
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_set, batch_size=64, shuffle=False)
val_loader   = DataLoader(val_set, batch_size=64, shuffle=False)

In [82]:
class ANN(nn.Module):
    def __init__(self, input_size = X_train.shape[1]):
        super().__init__()

        self.linear = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(128, 2)
        )
    def forward(self, x):
        x = self.linear(x)
        return x

In [83]:
model = ANN()
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=5)

In [84]:
best_para = 0.0
epochs = 20

for epoch in range(epochs):

    # Training
    model.train()
    running_loss = 0.0
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device) 
        optimizer.zero_grad()

        output = model(Xb)
        loss = criterion(output, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    avg_training_loss = running_loss / len(train_loader)


    # Testing
    model.eval()
    
    val_probs = []
    val_targets = []

    with torch.no_grad():
        test_running_loss = 0.0
        for Xb, yb in test_loader:
            Xb, yb = Xb.to(device), yb.to(device) 

            output = model(Xb)
            loss = criterion(output, yb)
            test_running_loss += loss.item()

            prob = torch.softmax(output, dim = 1)[:, 1]
            val_probs.extend(prob.cpu().numpy())
            val_targets.extend(yb.cpu().numpy())

        avg_testing_loss = test_running_loss / len(test_loader)
        auc = roc_auc_score(val_targets, val_probs)
        scheduler.step(auc)

    if auc > best_para:
        best_para = auc
        torch.save(model.state_dict(), 'best_model.pth')

    print(f"Epoch {epoch+1:02d}: "
          f"Train Loss: {avg_training_loss:.4f} | "
          f"Val Loss: {avg_testing_loss:.4f} | "
          f"Val AUC: {auc:.4f}")
            




Epoch 01: Train Loss: 0.3405 | Val Loss: 0.3587 | Val AUC: 0.7801
Epoch 02: Train Loss: 0.3158 | Val Loss: 0.3775 | Val AUC: 0.7181
Epoch 03: Train Loss: 0.3093 | Val Loss: 0.3883 | Val AUC: 0.7275
Epoch 04: Train Loss: 0.3059 | Val Loss: 0.3923 | Val AUC: 0.7269
Epoch 05: Train Loss: 0.3015 | Val Loss: 0.3522 | Val AUC: 0.7991
Epoch 06: Train Loss: 0.2998 | Val Loss: 0.3471 | Val AUC: 0.7940
Epoch 07: Train Loss: 0.2980 | Val Loss: 0.3365 | Val AUC: 0.8041
Epoch 08: Train Loss: 0.2969 | Val Loss: 0.4283 | Val AUC: 0.7295
Epoch 09: Train Loss: 0.2956 | Val Loss: 0.3175 | Val AUC: 0.8263
Epoch 10: Train Loss: 0.2947 | Val Loss: 0.3910 | Val AUC: 0.7522
Epoch 11: Train Loss: 0.2939 | Val Loss: 0.3373 | Val AUC: 0.7970
Epoch 12: Train Loss: 0.2927 | Val Loss: 0.3224 | Val AUC: 0.8255
Epoch 13: Train Loss: 0.2927 | Val Loss: 0.3978 | Val AUC: 0.7091
Epoch 14: Train Loss: 0.2918 | Val Loss: 0.4184 | Val AUC: 0.7213
Epoch 15: Train Loss: 0.2911 | Val Loss: 0.3661 | Val AUC: 0.7660
Epoch 16: 

KeyboardInterrupt: 

In [86]:
import lightgbm as lgb
import xgboost as xgb


lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'num_leaves': 127,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'lambda_l1': 0.5,
    'lambda_l2': 0.5,

    'verbose': -1,
    'seed': 42
}

train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

lgb_model = lgb.train(
    lgb_params, train_data, num_boost_round=500,
    valid_sets=[val_data],
    callbacks=[lgb.log_evaluation(period=100), lgb.early_stopping(50)]
)

y_pred_lgb_train = lgb_model.predict(X_train)  # train predictions
y_pred_lgb_test  = lgb_model.predict(X_test)   # test predictions (for AUC scoring)
y_pred_lgb_val   = lgb_model.predict(X_val_tree)  # val predictions (for submission)

# AUC should be on X_test not X_train (train AUC is always inflated)
lgb_auc = roc_auc_score(y_test, y_pred_lgb_test)

print(f"✅ LightGBM ROC-AUC: {lgb_auc:.6f}")
# all_predictions['LightGBM'] = {'train': y_pred_lgb_train, 'test': y_pred_lgb_test, 'auc': lgb_auc}

# feature_importance_lgb = pd.DataFrame({
#     'feature': X_train_processed.columns,
#     'importance': lgb_model.feature_importance()
# }).sort_values('importance', ascending=False)


Training until validation scores don't improve for 50 rounds
[100]	valid_0's auc: 0.909511
[200]	valid_0's auc: 0.912045
[300]	valid_0's auc: 0.912231
[400]	valid_0's auc: 0.912415
Early stopping, best iteration is:
[369]	valid_0's auc: 0.912466
✅ LightGBM ROC-AUC: 0.912466


In [87]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import roc_auc_score

# ── 1. LightGBM ───────────────────────────────────────────────────────────────
train_data = lgb.Dataset(X_train_tree, label=y_train)
val_data   = lgb.Dataset(X_test_tree,  label=y_test, reference=train_data)

lgb_model  = lgb.train(
    lgb_params, train_data, num_boost_round=500,
    valid_sets=[val_data],
    callbacks=[lgb.log_evaluation(100), lgb.early_stopping(50)]
)

lgb_test_probs = lgb_model.predict(X_test_tree)
lgb_val_probs  = lgb_model.predict(X_val_tree)
lgb_auc        = roc_auc_score(y_test, lgb_test_probs)
print(f"✅ LightGBM      AUC: {lgb_auc:.6f}")

pd.DataFrame({'id': df_val['id'], 'PitNextLap': lgb_val_probs})\
    .to_csv('lightgbm.csv', index=False)

# ── 2. XGBoost ────────────────────────────────────────────────────────────────
xgb_model = xgb.XGBClassifier(
    n_estimators          = 500,
    learning_rate         = 0.05,
    max_depth             = 6,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    reg_alpha             = 0.5,
    reg_lambda            = 0.5,
    eval_metric           = 'auc',
    early_stopping_rounds = 50,
    random_state          = 42,
    verbosity             = 0,
    n_jobs                = -1
)

xgb_model.fit(
    X_train_tree, y_train,
    eval_set=[(X_test_tree, y_test)],
    verbose=False
)

xgb_test_probs = xgb_model.predict_proba(X_test_tree)[:, 1]
xgb_val_probs  = xgb_model.predict_proba(X_val_tree)[:, 1]
xgb_auc        = roc_auc_score(y_test, xgb_test_probs)
print(f"✅ XGBoost       AUC: {xgb_auc:.6f}")

pd.DataFrame({'id': df_val['id'], 'PitNextLap': xgb_val_probs})\
    .to_csv('xgboost.csv', index=False)

# ── 3. HistGradientBoosting ───────────────────────────────────────────────────
hgb_model = HistGradientBoostingClassifier(
    max_iter            = 500,
    learning_rate       = 0.05,
    max_leaf_nodes      = 127,
    l2_regularization   = 0.5,
    early_stopping      = True,
    validation_fraction = 0.1,
    n_iter_no_change    = 50,
    random_state        = 42
)

hgb_model.fit(X_train_tree, y_train)

hgb_test_probs = hgb_model.predict_proba(X_test_tree)[:, 1]
hgb_val_probs  = hgb_model.predict_proba(X_val_tree)[:, 1]
hgb_auc        = roc_auc_score(y_test, hgb_test_probs)
print(f"✅ HistGB        AUC: {hgb_auc:.6f}")

pd.DataFrame({'id': df_val['id'], 'PitNextLap': hgb_val_probs})\
    .to_csv('histgb.csv', index=False)

# ── 4. Logistic Regression ────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler

scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_tree)
X_test_sc  = scaler.transform(X_test_tree)
X_val_sc   = scaler.transform(X_val_tree)

lr_model = LogisticRegression(
    C            = 0.1,
    max_iter     = 1000,
    class_weight = 'balanced',
    random_state = 42,
    n_jobs       = -1
)

lr_model.fit(X_train_sc, y_train)

lr_test_probs = lr_model.predict_proba(X_test_sc)[:, 1]
lr_val_probs  = lr_model.predict_proba(X_val_sc)[:, 1]
lr_auc        = roc_auc_score(y_test, lr_test_probs)
print(f"✅ LogisticReg   AUC: {lr_auc:.6f}")

pd.DataFrame({'id': df_val['id'], 'PitNextLap': lr_val_probs})\
    .to_csv('logistic_regression.csv', index=False)

# ── 5. Ridge ──────────────────────────────────────────────────────────────────
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_sc, y_train)

ridge_test_probs = np.clip(ridge_model.predict(X_test_sc), 0, 1)
ridge_val_probs  = np.clip(ridge_model.predict(X_val_sc),  0, 1)
ridge_auc        = roc_auc_score(y_test, ridge_test_probs)
print(f"✅ Ridge         AUC: {ridge_auc:.6f}")

pd.DataFrame({'id': df_val['id'], 'PitNextLap': ridge_val_probs})\
    .to_csv('ridge.csv', index=False)

# ── 6. Summary ────────────────────────────────────────────────────────────────
print("\n── AUC Summary ──")
results = {
    'LightGBM':    lgb_auc,
    'XGBoost':     xgb_auc,
    'HistGB':      hgb_auc,
    'LogisticReg': lr_auc,
    'Ridge':       ridge_auc,
}
for name, auc in sorted(results.items(), key=lambda x: -x[1]):
    print(f"  {name:<15} AUC: {auc:.6f}")

# ── 7. Blend all & export ─────────────────────────────────────────────────────
val_preds  = [lgb_val_probs, xgb_val_probs, hgb_val_probs, lr_val_probs, ridge_val_probs]
test_preds = [lgb_test_probs, xgb_test_probs, hgb_test_probs, lr_test_probs, ridge_test_probs]

# AUC-weighted blend
weights      = np.array(list(results.values()))
weights      = weights / weights.sum()
ensemble_val = sum(w * p for w, p in zip(weights, val_preds))
ensemble_auc = roc_auc_score(y_test, sum(w * p for w, p in zip(weights, test_preds)))
print(f"\n  {'Ensemble':<15} AUC: {ensemble_auc:.6f}")

pd.DataFrame({'id': df_val['id'], 'PitNextLap': ensemble_val})\
    .to_csv('ensemble.csv', index=False)

print("\n── Saved CSVs ──")
for name in ['lightgbm', 'xgboost', 'histgb', 'logistic_regression', 'ridge', 'ensemble']:
    print(f"  {name}.csv ✓")

Training until validation scores don't improve for 50 rounds
[100]	valid_0's auc: 0.909511
[200]	valid_0's auc: 0.912045
[300]	valid_0's auc: 0.912231
[400]	valid_0's auc: 0.912415
Early stopping, best iteration is:
[369]	valid_0's auc: 0.912466
✅ LightGBM      AUC: 0.912466
✅ XGBoost       AUC: 0.909036
✅ HistGB        AUC: 0.911865
✅ LogisticReg   AUC: 0.862735
✅ Ridge         AUC: 0.845211

── AUC Summary ──
  LightGBM        AUC: 0.912466
  HistGB          AUC: 0.911865
  XGBoost         AUC: 0.909036
  LogisticReg     AUC: 0.862735
  Ridge           AUC: 0.845211

  Ensemble        AUC: 0.901862

── Saved CSVs ──
  lightgbm.csv ✓
  xgboost.csv ✓
  histgb.csv ✓
  logistic_regression.csv ✓
  ridge.csv ✓
  ensemble.csv ✓
